In [1]:
from utils import *

In [2]:
g = 3
r = 3
d = 1 # dejar así
L = Lefschetz()
X = Curve("X", g)
H = CurveChow(X.name, X.g)
J = Jacobian(X).get_lambda_var(1)

In [3]:
# motive_H = (motive_generic(X, r, d)/J - L**2*X.get_lambda_var(2)).expand().simplify()
motive_X = gl(X, r)
# print(motive_H) 
motive_X

L**16 + L**6*λ2(X)**2 + X**2*(L**11 + L**3) + X*λ2(X)*(L**8 + L**5) + X*λ2(X)*(L**9 + L**4) + X*λ3(X)*(L**7 + L**5) + X*(L**13 + L**2) + X*(L**14 + L) + λ2(X)*(L**10 + L**4) + λ2(X)*(L**12 + L**2) + λ3(X)*(L**7 + L**6) + λ3(X)*(L**10 + L**3) + λ4(X)*(L**8 + L**4) + 1

---

In [5]:
def find_motive_X_low(obj: sp.Expr, X: Curve, coefficients: list[sp.Expr]):
    """
    return positive polynomial in X equal to motive (aquí la chicha), obj dado en suma positiva en h1
    """
    max_degree = X.g
    motive = 0
    remaining = obj
    for degree in range(max_degree, 0, -1):  # como ordenamos el graddo
        for i in range(degree, -1, -1):  # como ordenamos producto con el grado
            x_power_h = X.get_lambda_var(i)*X.get_lambda_var(degree)
            x_power_sym = sym_lambda(X, i)*sym_lambda(X, degree) 
            total_coef = 0
            for coef in coefficients:   # como ordenamos coeficientes
                candidate = (remaining - coef*x_power_h).expand()
                while not any(term.could_extract_minus_sign() for term in candidate.as_ordered_terms()):   # Cual es nuestra condición de parada            
                    total_coef += coef
                    candidate = (candidate - coef*x_power_h).expand()
            motive += total_coef*x_power_sym
            remaining = (remaining - total_coef*x_power_h).expand()
    if any(remaining.has(X.curve_chow.get_lambda_var(i)) for i in range(1, g+1)):
        return None
    return motive + remaining

def find_motive_X(obj: sp.Expr, X: Curve, coefficients: list[sp.Expr], d1=-1, d2=-1, switch=True):
    """
    return positive polynomial in X equal to motive (aquí la chicha), obj dado en suma positiva en h1
    """
    expanded_coefficients = [0] + coefficients + [coefficients[i] + coefficients[j] for i in range(len(coefficients)) for j in range(i+1)]
    if d1 == -1:
        d1 = 2*X.g
        d2 = d1
    if max(d1,d2) <= X.g:
        return find_motive_X_low(obj, X, coefficients)        
    x_power_h = X.get_lambda_var(d1)*X.get_lambda_var(d2)
    x_power_sym = sym_lambda(X, d1)*sym_lambda(X, d2)
    no_need_coefs = set()
    d1, d2 = d2, d1
    if not switch:
        if d2 == 0:
            d1 -= 1
            d2 = d1
        else:
            d2 -= 1
    for i, coef in enumerate(expanded_coefficients):
        if coef in no_need_coefs:
            continue
        rest = (obj-coef*x_power_h).expand()
        if not any(term.could_extract_minus_sign() for term in rest.as_ordered_terms()):
            motive_rest = find_motive_X(rest, X, coefficients, d1=d1, d2=d2, switch=not switch)   #repeticion de grupo
            if motive_rest != None:
                return coef*x_power_sym + motive_rest
        elif i <= len(coefficients):
            for exp_coef in expanded_coefficients[len(coefficients)+1:]:
                if coef in exp_coef.args:
                    no_need_coefs.add(exp_coef)
    return None


---

In [6]:
n = 21
coefficients = [L**i for i in range(n+1)]  
motive = find_motive_X_low(motive_H, X, coefficients)

In [24]:
print(motive)

L**6 + L**2*\lambda\!2(X) + X*(L**4 + L) + 1


In [25]:
motive

L**6 + L**2*\lambda\!2(X) + X*(L**4 + L) + 1

In [ ]:
compare(subs_H(motive, X), motive_H)

In [ ]:
motive_H